The Inference Code
Building a clean, interactive testing environment. Paste these code blocks into separate cells in your new notebook so you can run them one by one.

In [1]:
import torch
import pickle
import pandas as pd

# Load the embeddings we just trained
print("Loading LightGCN memory...")
with open('models/lightgcn_embeddings.pkl', 'rb') as f:
    data = pickle.load(f)

# Convert the NumPy arrays back to PyTorch Tensors for lightning-fast math
user_embeddings = torch.tensor(data['user_embeddings'])
movie_embeddings = torch.tensor(data['movie_embeddings'])

user_to_idx = data['user_to_idx']
title_to_idx = data['title_to_idx']
idx_to_title = data['idx_to_title']

print(f"Loaded {len(user_embeddings)} users and {len(movie_embeddings)} movies.")

Loading LightGCN memory...
Loaded 5000 users and 4223 movies.


We just do a pure linear algebra dot product to find the closest vectors.

In [2]:
def get_graph_recommendations(username, top_k=5):
    if username not in user_to_idx:
        return "User not found!"
        
    u_idx = user_to_idx[username]
    
    # 1. Grab the specific user's brain (vector)
    u_emb = user_embeddings[u_idx]
    
    # 2. Matrix Multiplication: Dot product this user against EVERY movie instantly
    scores = torch.matmul(movie_embeddings, u_emb)
    
    # 3. Sort and get the highest scores
    top_scores, top_indices = torch.topk(scores, top_k)
    
    print(f"🎯 Graph Recommendations for {username}:\n")
    for i in range(top_k):
        m_idx = top_indices[i].item()
        score = top_scores[i].item()
        title = idx_to_title[m_idx]
        print(f"{i+1}. {title} (Match Score: {score:.2f})")

Test It Out!
We generated mock users earlier, let's grab one of them and see what the graph predicts they will like.

In [4]:
# Try changing this to cine_fan_10, cine_fan_42, etc.
get_graph_recommendations('cine_fan_4000')

🎯 Graph Recommendations for cine_fan_4000:

1. Ajami (Match Score: 0.04)
2. Starship Troopers (Match Score: 0.04)
3. Jack and Jill (Match Score: 0.03)
4. The Ghost Writer (Match Score: 0.03)
5. Akira (Match Score: 0.03)
